In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/archive.zip" .
!unzip -q archive.zip

In [3]:
import torch.nn as nn
import torch
import torchvision
from torchvision import datasets
from torchvision.transforms import v2
import torch.optim as optim

In [17]:
train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomCrop(32, padding=4),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 128

trainset = datasets.ImageFolder(
    root="cifar10/train",
    transform=train_transform
)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = datasets.ImageFolder(
    root="cifar10/test",
    transform=test_transform
)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [18]:
images, labels = next(iter(trainloader))

print(images.shape)
print(labels)

torch.Size([128, 3, 32, 32])
tensor([2, 2, 0, 2, 2, 9, 0, 8, 5, 4, 5, 8, 8, 3, 5, 6, 7, 4, 2, 7, 9, 9, 8, 5,
        0, 9, 9, 5, 8, 4, 0, 8, 3, 5, 9, 7, 6, 8, 7, 3, 1, 8, 6, 6, 9, 6, 4, 5,
        0, 0, 0, 1, 7, 0, 3, 3, 9, 4, 3, 5, 1, 0, 6, 1, 4, 2, 4, 5, 1, 0, 3, 1,
        0, 1, 3, 2, 3, 8, 1, 2, 1, 4, 3, 5, 0, 3, 1, 2, 2, 8, 9, 2, 5, 6, 4, 2,
        2, 8, 3, 4, 6, 4, 7, 5, 9, 6, 6, 4, 7, 9, 3, 0, 1, 2, 1, 6, 9, 8, 1, 1,
        2, 9, 5, 7, 0, 7, 9, 3])


In [19]:
class CNN(nn.Module):
  def __init__(self):
        super().__init__()
        self.fwpass = nn.Sequential(
            nn.Conv2d(3,32,3),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32,64,3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64,128,3),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128*11*11,10)
        )

  def forward(self, x):
      logits = self.fwpass(x)
      return logits

In [20]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(device)
net = CNN().to(device)

cuda


In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.005, momentum=0.9)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=8, gamma=0.5)

In [22]:
net.train()
for epoch in range(25):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)


        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 100 == 99:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0
    scheduler.step()
print('Finished Training')

[1,   100] loss: 2.041
[1,   200] loss: 1.623
[1,   300] loss: 1.448
[2,   100] loss: 1.218
[2,   200] loss: 1.179
[2,   300] loss: 1.147
[3,   100] loss: 1.037
[3,   200] loss: 1.027
[3,   300] loss: 0.992
[4,   100] loss: 0.923
[4,   200] loss: 0.919
[4,   300] loss: 0.923
[5,   100] loss: 0.891
[5,   200] loss: 0.852
[5,   300] loss: 0.848
[6,   100] loss: 0.811
[6,   200] loss: 0.817
[6,   300] loss: 0.814
[7,   100] loss: 0.785
[7,   200] loss: 0.764
[7,   300] loss: 0.783
[8,   100] loss: 0.746
[8,   200] loss: 0.745
[8,   300] loss: 0.722
[9,   100] loss: 0.688
[9,   200] loss: 0.665
[9,   300] loss: 0.652
[10,   100] loss: 0.645
[10,   200] loss: 0.650
[10,   300] loss: 0.637
[11,   100] loss: 0.640
[11,   200] loss: 0.623
[11,   300] loss: 0.635
[12,   100] loss: 0.625
[12,   200] loss: 0.623
[12,   300] loss: 0.619
[13,   100] loss: 0.616
[13,   200] loss: 0.601
[13,   300] loss: 0.619
[14,   100] loss: 0.607
[14,   200] loss: 0.605
[14,   300] loss: 0.610
[15,   100] loss: 0

In [23]:
net.eval()
total = 0
correct = 0
with torch.no_grad():
  for images,labels in testloader:
    images, labels = images.to(device), labels.to(device)
    outputs = net(images)
    predictions = torch.argmax(outputs, dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)
accuracy = correct / total
print(accuracy)

0.8256
